In [1]:
import furuta_systems
import pandas as pd
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import furuta_params
import furuta_trajectory_gen
import copy
import scipy

# Versuchsdurchführung

### Aufbauend auf dem nun vollständigen Parametersatz, lässt sich der modellbasierte Entwurf für die Steuerung für die stabile Ruhelage theta2 = 0° testen.

In [2]:
params_all = furuta_params.FurutaParams()
# params_all.mu_H1 = 
# params_all.mu_H2 =
# params_all.mu_V1 = 
# params_all.mu_V2 = 
# params_all.L1 = 
# params_all.L2 = 
# params_all.lp = 
# params_all.mS = 
# params_all.mP = 
# params_all.J1_hat = 
params_all.epsilon = 0.01
params_all.tau_max = 4.0
    
params_muH_eq0 = copy.copy(params_all)
params_muH_eq0.mu_H1 = 0.0
params_muH_eq0.mu_H2 = 0.0

# Stabile Ruhelage

### 4.4 Berechnen Sie für die stabile Ruhelage die Systemmatrix A und den Eingangsvektor b der linearen Darstellung.

In [3]:
params_steuerung = params_muH_eq0

stable_op = (0.0, 0.0, 0.0, 0.0, 0.0)
A, b = furuta_systems.get_linear_furuta(params_steuerung, stable_op)
print("A = \n" + 
    np.array2string(
    A,
    precision=3,
    suppress_small=True,
    max_line_width=120
))
print("b = \n" + 
    np.array2string(
    b,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

A = 
[[  0.      1.      0.      0.   ]
 [  0.    -17.516   1.075   0.   ]
 [  0.      0.      0.      1.   ]
 [  0.     16.969 -62.354  -0.026]]
b = 
[[  0.   ]
 [ 34.406]
 [  0.   ]
 [-33.331]]


### 4.5 Berechnen Sie die Transformationsmatrix T zur Überführung auf RNF und die korrespondierende Systemmatrix.

In [4]:
A_RNF, Q = furuta_systems.get_RNF_furuta(params_steuerung, stable_op)

print("A_RNF = \n" + 
    np.array2string(
    A_RNF,
    precision=5,
    suppress_small=True,
    max_line_width=120
))

print("Q = \n" + 
    np.array2string(
    Q,
    precision=10,
    suppress_small=True,
    max_line_width=120
))

A_RNF = 
[[    0.          1.          0.          0.     ]
 [    0.          0.          1.          0.     ]
 [    0.          0.          0.          1.     ]
 [    0.      -1073.96023   -62.7975    -17.54192]]
Q = 
[[ 0.0004740399 -0.0000001958  0.0004893264 -0.0000002021]
 [ 0.            0.0004740399  0.0000123896  0.0004893315]
 [ 0.           -0.           -0.0300021381 -0.          ]
 [ 0.            0.            0.           -0.0300021381]]


### 4.6 Testen Sie die Steuerung anhand einer Überführung von der Ruhelage x(t1) =(0°, °/s, 0°, 0°/s) auf x(t2) = (140°, 0°/s, 0°, 0°/s) für die Zeiten t1 = 0 s und t2 =2.5 s.


### a und b Koeffizienten:

In [5]:
cT = np.array([[1, 0, 0, 0]])
a0, a1, a2, a3 = furuta_systems.get_a_coefficients_furuta(params_steuerung, stable_op)
b0, b1, b2, b3 = furuta_systems.get_b_coefficients_furuta(cT, params_steuerung, stable_op)
a = np.array([a0, a1, a2, a3])
b = np.array([b0, b1, b2, b3])

print("a = " + 
    np.array2string(
    a,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

print("b = " + 
    np.array2string(
    b,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

a = [  -0.    1073.96    62.798   17.542]
b = [2109.527    0.871   34.406   -0.   ]


### Trajectorien in csv abspeichern

In [6]:
y0 = 0.0
yT = (140.0/360.0) * 2 * np.pi
t0 = 0
t1 = 2.5

eta0 = y0/b0
etaT = yT/b0

eta_ref, eta_d1_ref, eta_d2_ref, eta_d3_ref, eta_d4_ref = \
furuta_trajectory_gen.generate_only_eta_trajectory(eta0, etaT, t0, t1, params_steuerung, stable_op)

N = 1000               # Anzahl Abtastpunkte
t_vec = np.linspace(t0, t1, N)

# Funktionen auswerten
eta_vec     = np.array([eta_ref(t) for t in t_vec])
eta_d1_vec  = np.array([eta_d1_ref(t) for t in t_vec])
eta_d2_vec  = np.array([eta_d2_ref(t) for t in t_vec])
eta_d3_vec  = np.array([eta_d3_ref(t) for t in t_vec])
eta_d4_vec  = np.array([eta_d4_ref(t) for t in t_vec])

# In DataFrame packen
df = pd.DataFrame({
    "t": t_vec,
    "eta": eta_vec,
    "eta_d1": eta_d1_vec,
    "eta_d2": eta_d2_vec,
    "eta_d3": eta_d3_vec,
    "eta_d4": eta_d4_vec,
})

# Als CSV exportieren
df.to_csv("furuta_trajectory_stable.csv", index=False)

### 4.7 Berechnen Sie mit den bekannten Parametern eine passende Reglerverstärkung k.

In [7]:
params_regelung = params_muH_eq0

#Regler Parameter (harmonischer Oszillator)
omega0_scaler = 1.0 #omega0 = max. eigenwert von A
D = -0.5
D_down = 1.01 * D
A, b = furuta_systems.get_linear_furuta(params_regelung, stable_op)
eigvals = np.linalg.eigvals(A)
omega0 = float(omega0_scaler * np.max(np.abs(eigvals)))

imag = np.sqrt(1.0 - D**2)
k_res = scipy.signal.place_poles(A, b, (D*omega0, D_down*omega0, omega0*(D+1j*imag), omega0*(D-1j*imag)))
k = k_res.gain_matrix

print("k = " + 
    np.array2string(
    k,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

k = [[11.141  2.663 -7.209  2.224]]


### 4.8 Testen Sie die Steuerung mit dem Zustandsregler anhand einer Überführung von der Ruhelage x(t1) = (0°, °/s, 0°, 0°/s) auf x(t2) = (140°, 0°/s, 0°, 0°/s) für die Zeiten t1 = 0 s und t2 = 2.5 s. Stören Sie den Prozess.

# Instabile Ruhelage

### Wiederholen Sie nun ihre vorangegangenen Schritte für die instabile Ruhelage theta2 = 180°.

### 4.9 Berechnen Sie zunächst die Systemmatrix A und den Eingangsvektor u, die Transformationsmatrix T auf RNF, sowie die damit korrespondierende Systemmatrix.

In [8]:
params_steuerung = params_muH_eq0

theta2_unstable_op = np.pi

unstable_op = (0.0, 0.0, theta2_unstable_op, 0.0, 0.0)
A, b = furuta_systems.get_linear_furuta(params_steuerung, unstable_op)
print("A = \n" + 
    np.array2string(
    A,
    precision=3,
    suppress_small=True,
    max_line_width=120
))
print("b = \n" + 
    np.array2string(
    b,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

A_RNF, Q = furuta_systems.get_RNF_furuta(params_steuerung, unstable_op)

print("A_RNF = \n" + 
    np.array2string(
    A_RNF,
    precision=5,
    suppress_small=True,
    max_line_width=120
))

print("Q = \n" + 
    np.array2string(
    Q,
    precision=20,
    suppress_small=True,
    max_line_width=120
))

A = 
[[  0.      1.      0.      0.   ]
 [  0.    -17.516   1.075  -0.   ]
 [  0.      0.      0.      1.   ]
 [  0.    -16.969  62.354  -0.026]]
b = 
[[ 0.   ]
 [34.406]
 [ 0.   ]
 [33.331]]
A_RNF = 
[[   0.         1.         0.         0.     ]
 [   0.         0.         1.         0.     ]
 [   0.         0.         0.         1.     ]
 [   0.      1073.96023   61.9105   -17.54192]]
Q = 
[[-0.00047403989806320046 -0.00000019575745249154  0.0004893366240137266   0.00000020207220902353]
 [ 0.                     -0.00047403989806320046  0.00001238955231575504  0.0004893315076781423 ]
 [ 0.                      0.00000000000000000081  0.030002138064516105   -0.00000000000000000085]
 [ 0.                     -0.00000000000000002266 -0.00000000000000005291  0.030002138064516105  ]]


### 4.10 Berechnen Sie mit den bekannten Parametern eine passende Reglerverstärkung k.

In [13]:
params_regelung = params_muH_eq0
A, b = furuta_systems.get_linear_furuta(params_regelung, unstable_op)
Q = np.array([ [10, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1] ])
R = np.array([[5]])

from scipy.linalg import solve_continuous_are
P = solve_continuous_are(A, b, Q, R)
k = np.linalg.inv(R) * (b.T @ P)

print("k = \n" + 
    np.array2string(
    k,
    precision=5,
    suppress_small=True,
    max_line_width=120
))

k = 
[[-1.41421 -1.61618 19.02098  2.4616 ]]


### 4.11 Testen Sie die Steuerung mit dem Zustandsregler anhand einer Überführung von der Ruhelage x(t1) = (0°, 0°/s, 180°, 0°/s) auf x(t2) = (140°, 0°/s, 180°, 0°/s) für die Zeiten t1 = 10 s und t2 = 12.5 s. Stellen Sie das Pendel händisch in die instabile Ruhelage und stören Sie zusätzlich den Prozess.

### a und b Koeffizienten:

In [10]:
cT = np.array([[1, 0, 0, 0]])
a0, a1, a2, a3 = furuta_systems.get_a_coefficients_furuta(params_steuerung, unstable_op)
b0, b1, b2, b3 = furuta_systems.get_b_coefficients_furuta(cT, params_steuerung, unstable_op)
a = np.array([a0, a1, a2, a3])
b = np.array([b0, b1, b2, b3])

print("a = " + 
    np.array2string(
    a,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

print("b = " + 
    np.array2string(
    b,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

a = [   -0.    -1073.96    -61.911    17.542]
b = [-2109.527     0.871    34.406     0.   ]


### Trajectorien in csv abspeichern

In [12]:
y0 = 0.0
yT = (140.0/360.0) * 2 * np.pi
t0 = 10
t1 = 12.5

eta0 = y0/b0
etaT = yT/b0

eta_ref, eta_d1_ref, eta_d2_ref, eta_d3_ref, eta_d4_ref = \
furuta_trajectory_gen.generate_only_eta_trajectory(eta0, etaT, t0, t1, params_steuerung, unstable_op)

N = 1000               # Anzahl Abtastpunkte
t_vec = np.linspace(t0, t1, N)

# Funktionen auswerten
eta_vec     = np.array([eta_ref(t) for t in t_vec])
eta_d1_vec  = np.array([eta_d1_ref(t) for t in t_vec])
eta_d2_vec  = np.array([eta_d2_ref(t) for t in t_vec])
eta_d3_vec  = np.array([eta_d3_ref(t) for t in t_vec])
eta_d4_vec  = np.array([eta_d4_ref(t) for t in t_vec])

# In DataFrame packen
df = pd.DataFrame({
    "t": t_vec,
    "eta": eta_vec,
    "eta_d1": eta_d1_vec,
    "eta_d2": eta_d2_vec,
    "eta_d3": eta_d3_vec,
    "eta_d4": eta_d4_vec,
})

# Als CSV exportieren
df.to_csv("furuta_trajectory_unstable.csv", index=False)